# 持久化检查点（Checkpointer）与状态恢复机制
虽然搭建了具备“大脑”（LLM）、“手脚”（Tools）、“内存”（State）以及“循环路由”（Conditional Edges）的 ReAct 智能体 。但在企业真实落地场景中，仅仅能够运行 ReAct 循环是远远不够的。

当 Agent 在生产环境中遇到用户刷新页面、服务重启、需要跨天连续对话，或者需要进行人工干预审查（Human-in-the-loop） 时，传统的内存型 Agent 会立刻失忆或无法协同。
LangGraph 中极具生产价值的核心特性——持久化检查点（Checkpointer）与状态恢复机制 。

## Memory Checkpoint 与持久化机制
1. 理解 Checkpointer 工作原理：搞懂 LangGraph 如何在节点执行的边界（Checkpoint）自动打快照（Snapshot）。
2. 掌握 Thread ID（会话线程隔离）：实现多用户、多会话之间的状态物理隔离与持久化存储 。
3. 实现历史状态回溯与修改：学会提取特定时间点的 State 历史，并实现“时间旅行”与重试纠错。

#### 什么是 Checkpointer（检查point）？
在 LangGraph 中，Checkpointer 相当于系统的“自动存档点” 。每当一个 Node 执行完毕、或者图进入等待/路由阶段时，Checkpointer 会自动将当前的全局 `State`、节点执行记录以及元数据写入持久化存储介质中。

生产级 Checkpointer 的四大核心作用：
1. 多轮对话持久化：将对话历史、上下文变量持久化到数据库（如 SQLite, PostgreSQL, Redis），无需自己手动维护 Session DB 。
2. 容错恢复：如果服务突然宕机或模型 API 超时报错，重新启动后可以从上一个 Checkpoint 无缝继续运行 。
3. 人机协同（Human-in-the-loop）：让 Agent 在执行高危操作（如转账、删除数据库）前中断（Interrupt），等待人类在前端确认后再恢复执行 。
4. 状态回溯与分支调试：支持拉取历史状态，甚至覆盖过去某一步的 State 并重新分支运行（Time Travel） 。

#### Thread ID 与配置（RunnableConfig）
在带有 Checkpointer 的 LangGraph 应用中，每次调用 `invoke` 或 `stream` 时，都必须传入一个 `config` 字典，其中最重要的参数就是 `thread_id` ：

In [ ]:
config = {"configurable": {"thread_id": "user_123_session_001"}}

* `thread_id`：标识一条独立的对话或任务执行线。相同的 `thread_id` 会共享并累加相同的 Checkpoint 历史状态 ；不同的 `thread_id` 则完全隔离，互相不干扰 。

使用 LangGraph 提供的 MemorySaver（用于内存级持久化演示）及 SqliteSaver 思想，为上一节课手写的 ReAct 智能体加上持久化存档能力 。

In [ ]:
from typing import Annotated, Literal, TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool

# 导入 LangGraph 检查点管理器 (Checkpointer)
from langgraph.checkpoint.memory import MemorySaver

# ================= 1. 定义状态与工具 =================
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

@tool
def multiply(a: int, b: int) -> int:
    """计算两个整数的乘积。"""
    return a * b

tools = [multiply]
tool_node = ToolNode(tools)
llm = ChatOpenAI(model="gpt-4o", temperature=0).bind_tools(tools)

# ================= 2. 定义节点与路由 =================
def call_model(state: AgentState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

def should_continue(state: AgentState) -> Literal["action", "__end__"]:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "action"
    return "__end__"

# ================= 3. 构建图结构 =================
workflow = StateGraph(AgentState)
workflow.add_node("agent", call_model)
workflow.add_node("action", tool_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue)
workflow.add_edge("action", "agent")

# ================= 4. 绑定 Checkpointer 编译应用 =================
# 初始化检查点存储器 (生产环境中可换成 SqliteSaver / PostgresSaver)
checkpointer = MemorySaver()

# 在 compile 时注入 checkpointer！
app = workflow.compile(checkpointer=checkpointer)

# ================= 5. 多轮对话与会话隔离测试 =================
# 定义会话配置 A
config_thread_a = {"configurable": {"thread_id": "session_A"}}
# 定义会话配置 B
config_thread_b = {"configurable": {"thread_id": "session_B"}}

print("=== 1. 在 Session A 中提供信息 ===")
res1 = app.invoke(
    {"messages": [HumanMessage(content="你好，我叫小明，我是一名 Python 工程师。")]},
    config=config_thread_a
)
print("🤖 Session A 回复:", res1["messages"][-1].content)

print("\n=== 2. 在 Session B 中询问身份 (预期: 不知道) ===")
res2 = app.invoke(
    {"messages": [HumanMessage(content="你知道我是谁吗？")]},
    config=config_thread_b
)
print("🤖 Session B 回复:", res2["messages"][-1].content)

print("\n=== 3. 回到 Session A 中追问 (预期: 记得是小明) ===")
res3 = app.invoke(
    {"messages": [HumanMessage(content="你还记得我叫什么名字和做什么工作吗？")]},
    config=config_thread_a
)
print("🤖 Session A 回复:", res3["messages"][-1].content)

print("\n=== 4. 检查 Session A 的物理快照历史 (Checkpoint History) ===")
# 使用 app.get_state_history 可以查阅指定 thread 的所有历史快照
state_history = list(app.get_state_history(config_thread_a))
print(f"📊 Session A 至今共生成了 {len(state_history)} 个快照点。")

1. 上下文爆炸与快照裁剪：在极长的多轮对话场景中，`messages` 列表随着追加会越来越长 。虽然 Checkpointer 帮你保存了全部状态，但直接将全量消息喂给大模型会导致 Context Window 溢出或 Token 费用飙升 。
    * 思考：在 LangGraph 的节点中，你会采用什么策略或内置方法（例如 `RemoveMessage` 或滑动窗口总结）来保持全局 Checkpoint 历史完整的同时，精简喂给 LLM 的消息上下文？

2. 生产环境持久化选择：内存级别的 `MemorySaver` 在服务重启后数据就会丢失 。如果让你将上述系统部署到生产环境，且需要支持分布式部署（多实例扩展），你会选择哪种存储后端（如 Redis / PostgreSQL），并如何处理高并发下的 Checkpoint 写入竞争问题？

## 安全与合规关卡——人机协同（Human-in-the-Loop, HITL）机制
当 Agent 拥有了调用外部工具（执行 SQL 删除操作、发起资金转账、发送企业邮件、部署线上代码）的能力时，全自动执行可能带来不可逆的严重后果。如何在 Agent 决策路径中植入人类审批、干预纠错与状态重写，是构建可信 AI 系统的必修课 。


#### LangGraph 的 Human-in-the-Loop 与时间旅行
1. 掌握中断（Interrupt）机制：学会使用 `interrupt_before` / `interrupt_after` 让 Agent 在敏感节点自动暂停并挂起状态。
2. 实现人类审批与人工干预：学会通过 `update_state` 注入人类指导意见或修正 Agent 的决策历史。
3. 实现状态重写与“时间旅行（Time Travel）”：理解如何从任意历史 Checkpoint 分支重新运行任务。

#### Human-in-the-Loop 的三大应用场景
在 LangGraph 中，人机协同（HITL）并不是简单的 Python `input()` 阻塞，而是依托 Checkpointer 状态持久化实现的一种无状态、可异步的挂起与恢复机制：
1. 人工审批（Approval）：Agent 生成了高危工具调用的参数（例如 `transfer(amount=100000)`），系统暂停并通知管理员。只有当管理员审核通过后，流程才恢复执行。
2. 输入补全/纠错（Input Editing）：当 Agent 提取的实体参数错误或不完整时，人类可以动态修改当前 State 中的参数，再让 Agent 沿着新状态继续运行。
3. 状态回溯（Time Travel）：当 Agent 在第 5 步走入死胡同，人类可以直接将状态回退到第 2 步，修改其中的错误回复，强行纠正 Agent 的执行轨迹。

#### 带“人工审核与状态修正”的转账 Agent
下面我们实现一个包含敏感操作（转账工具）的 Agent。当大模型判定需要调用 `transfer_money` 工具时，系统会在工具执行前自动中断，等待人类审查。人类可以选择“赞成并继续”或“修改转账金额后再继续”。

In [ ]:
from typing import Annotated, Literal, TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver

# ================= 1. 定义状态与敏感工具 =================
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

@tool
def transfer_money(account: str, amount: float) -> str:
    """向指定账户转账。注意：这是高风险敏感操作！"""
    return f"成功向账户 {account} 转账 {amount} 元。"

tools = [transfer_money]
tool_node = ToolNode(tools)
llm = ChatOpenAI(model="gpt-4o", temperature=0).bind_tools(tools)

# ================= 2. 定义节点与路由 =================
def call_model(state: AgentState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

def should_continue(state: AgentState) -> Literal["action", "__end__"]:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "action"
    return "__end__"

# ================= 3. 构建 Graph 架构 =================
workflow = StateGraph(AgentState)
workflow.add_node("agent", call_model)
workflow.add_node("action", tool_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue)
workflow.add_edge("action", "agent")

checkpointer = MemorySaver()

# 🔑 关键点：在 compile 时，使用 interrupt_before 参数指定在执行 action 节点前强制暂停！
app = workflow.compile(
    checkpointer=checkpointer,
    interrupt_before=["action"]  # 在进入工具节点前挂起系统
)

# ================= 4. 人机协同（HITL）交互演练 =================
config = {"configurable": {"thread_id": "transfer_session_001"}}

print("=== 第一阶段：用户发起转账请求 ===")
# 用户发送转账指令
initial_input = {"messages": [HumanMessage(content="帮我给张三（账号：6222001）转账 10000 元")]}

# 运行图（Agent 会生成 call_tool 消息，但在进入 action 节点前会被中断）
for event in app.stream(initial_input, config):
    print("事件流输出:", event)

# 获取当前图的状态
current_state = app.get_state(config)
print("\n⏸️ 当前系统运行状态:")
print("下一个即将执行的节点 (next):", current_state.next)

# 打印 LLM 生成的转账意图，供人类审批
last_ai_msg = current_state.values["messages"][-1]
print("🤖 Agent 计划调用的工具及参数:", last_ai_msg.tool_calls)

# ================= 5. 第二阶段：人类人工审查与决策 =================
print("\n" + "="*40)
print("🛡️ 人类审查员干预阶段：检测到高风险转账！")

# 场景：人类审查员认为转账金额过大，干预修改转账金额为 5000 元
print("⚠️ 审查员操作：将转账金额修改为 5000 元...")

# 提取原有工具调用的 ID
original_tool_call = last_ai_msg.tool_calls[0]
updated_tool_call = {
    "name": original_tool_call["name"],
    "args": {"account": "6222001", "amount": 5000.0}, # 修改参数金额
    "id": original_tool_call["id"]
}

# 构造修正后的 AIMessage 覆盖旧状态
new_ai_msg = AIMessage(
    content=last_ai_msg.content,
    tool_calls=[updated_tool_call],
    id=last_ai_msg.id # 保留相同的消息 ID 以实现替换
)

# 使用 update_state 更新特定 Checkpoint 的状态
app.update_state(config, {"messages": [new_ai_msg]})

print("✅ 状态修改成功，恢复 Agent 流程运行...\n")

# ================= 6. 第三阶段：批准并恢复运行 =================
# 传入 None 作为输入，告诉 LangGraph 从上一次中断的位置恢复运行
for event in app.stream(None, config):
    print("恢复运行事件流:", event)

# 查看最终结果
final_state = app.get_state(config)
print("\n🏁 最终对话结果:")
for msg in final_state.values["messages"]:
    print(f"[{msg.__class__.__name__}]: {msg.content}")

1. `interrupt_before` / `interrupt_after`
    * 在 `compile()` 时定义，指定进入某个特定 Node（如 `action` 工具节点）前或后强行将线程暂停 。
    * 暂停时，系统会将此时的状态序列化并写入数据库（Checkpointer）中，应用服务可以安全关闭或等待 Webhook 回调 。

2. `app.update_state(config, values)`
    * 用于在图暂停期间由外部（人类 UI 界面、管理员系统）向 State 中注入新信息或覆盖原有消息。它相当于允许人类手动插入或重写图的 Memory 快照

3. `app.stream(None, config)` 恢复运行
    * 当接收到人类的批准信号后，调用 `invoke(None, config)` 或 `stream(None, config)`，LangGraph 会自动载入最新的 Checkpoint 快照，沿着 `current_state.next` 记录的待办节点继续向下流动

4. 在全自动多角色 Agent 团队场景下（例如开发 Agent、测试 Agent、运维 Agent），如果在部署上线节点引入 Human-in-the-Loop 审批，你将如何构建前端与 LangGraph 后端协同的异步 Webhook 通知机制？
5. 如果人类审批者拒绝了 Agent 的工具调用请求，你应该如何修改 State 中的 `messages`（例如向 State 中追加一条解释拒绝理由的 `ToolMessage`），以引导 Agent 重新思考并换一种无害的方式解决问题？